# real-time-lab (v10) — กล้อง → ท่ามือ · ใบหน้า · เสียง แบบสด

ทดสอบความ real-time และ performance ของ **v10** ในเครื่องนี้ (ไม่ต้องเปิด Docker) แยกเป็น 3 ส่วน + ภาพรวม:

| ส่วน | ทำอะไร | ดูอะไร |
|---|---|---|
| **4. Sign** | ทำท่าต่อเนื่อง → คำขึ้นระหว่างทำท่า: **เทา = partial** (อาจเปลี่ยน) → **ขาว = ยืนยันแล้ว** (ไม่แก้ย้อนหลัง) | latency ต่อคำ (ท่าจบ → คำยืนยันขึ้นจอ) p50/p95, fps, ms/เฟรมของ extractor, flicker |
| **5. Face** | ไม่ต้องทำท่า — ทำสีหน้า → อารมณ์ + เหตุผลภาษาไทย ทุก ~1 วิ | อารมณ์ 7 แบบ, ความมั่นใจ, คำอธิบาย (เช่น "มุมปากยกขึ้น (ยิ้ม)") |
| **6. Speech** | ประโยคทดสอบ (รวมท่าที่อ่านไม่ได้ `?`) → เสียง | การเว้นจังหวะ/คำลังเลแบบมนุษย์ ("ฉัน… อืม… พ่อ") แทนการข้ามคำ, เวลาสังเคราะห์ |
| **7. Overall** | ยกมือ → ทำท่าทั้งประโยค → วางมือ → ประโยค + อารมณ์ + **ปุ่มเล่นเสียง** | ผลแบบ production ทุกขั้น + เวลาแต่ละขั้น |

**Tier ของ extractor** (ข้อสรุปจาก extractor study ของ v10 — `result_reporting/result_v10.md` §2):
* **B = MediaPipe Holistic** (ค่าเริ่มต้น, CPU, ทุกเครื่องรวม Mac / ไม่มี GPU, โมเดลเดียวกับเบราว์เซอร์) — ชนะ RTMW ในการวัดมือบนข้อมูลของเรา
* **A = RTMW-X whole-body** (ทดลอง, ต้องมี `onnxruntime-gpu` ใน kernel — kernel `490-sign` มีแค่ CPU → ~70–100 ms/เฟรม ช้าเกิน real-time) — ไม่มี blendshape ของหน้า → ส่วน Face/อารมณ์ใช้ไม่ได้ใน tier A

Kernel: **490-sign** · กด **q** ในหน้าต่างกล้องเพื่อหยุดแต่ละส่วน (หรือ ■ Interrupt)

## 1. ตั้งค่า

In [1]:
import os, sys, time, json
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "modules").is_dir(), "เปิด notebook จาก root ของ repo"
sys.path.insert(0, str(ROOT))

MODEL_DIR = ROOT / "models" / "model-v10"        # forward v10 (rollback: ROOT / "models" / "model-v9")
FACE_DIR = ROOT / "models" / "face-model"        # face emotion model
SPEECH_DIR = ROOT / "models" / "speech-model-v2" # voice v2: emotion references + fillers (อืม / เอ่อ …)
os.environ["THAISLM_FACE_MODEL"] = str(FACE_DIR)

SOURCE = 0             # 0 = กล้องตัวแรก · หรือ path วิดีโอ เช่น "data_test/ฉันรักเพื่อน.mp4" (เล่นตามเวลาจริง ใช้วัด latency ได้เหมือนกล้อง)
TIER = "B"             # B = MediaPipe Holistic (CPU, ค่าเริ่มต้น) · A = RTMW-X whole-body (ทดลอง, ต้องมี GPU ใน onnxruntime)
DEVICE = "auto"        # ONNX: auto (CUDA → DirectML → CPU) · cpu · cuda · dml · coreml
VOCAB = "conversation" # ~330 คำสนทนา (เหมือนเว็บ) | full
ALPHABET = "en"        # สะกดนิ้ว en | th | both | off
SHOW_WINDOW = True     # หน้าต่าง OpenCV (กด q เพื่อหยุด)

# เสียง
SPEECH_ENGINE = "auto"            # auto = เสียง F5 ถ้า speech server (cell 6.0) เปิดอยู่ ไม่งั้นเสียง CPU (MMS) ใน kernel นี้ · cpu · server
SPEECH_SERVER = "http://127.0.0.1:8011"
VOICE = "female"                  # female | male
DISFLUENCY = "filler"             # filler = เว้นจังหวะ + คำลังเล (v10) · pause = เว้นจังหวะอย่างเดียว · skip = ข้ามแบบ v9
SPEAK_UNCERTAIN = False           # True = พูดคำ "อาจหมายถึง" ที่ P ≥ 0.5 ด้วย

# โหมดประโยค (ส่วน 7) — ค่าเดียวกับหน้าเว็บ
LIVE_PREROLL_MS, LIVE_UP_MS, END_MS, LIVE_MAX_MS, LIVE_MIN_MS, LIVE_MIN_UP_MS = 500, 250, 1500, 20000, 1200, 600
print("model:", MODEL_DIR.name, "| tier:", TIER, "| source:", SOURCE)

model: model-v10 | tier: B | source: 0


## 2. โหลดโมเดล

In [2]:
import numpy as np
from modules.pipeline import SignPipeline
from modules.utils import read_json

t0 = time.perf_counter()
pipe = SignPipeline(MODEL_DIR, backend="onnx", device=DEVICE, vocab=VOCAB, use_llm=False)   # LLM ปิด: ประโยคจาก rule
STREAM = dict(step_s=0.5, lookahead_s=0.8, stable_n=1)                                    # v9
STREAM.update(read_json(MODEL_DIR / "spotting.json").get("stream", {}))                   # v10: เลือกบน tune (decision stream)
print(f"{pipe.cfg.get('version')} · providers {pipe.be.providers} · face model {'on' if pipe.face_model is not None else 'off'}"
      f" · streaming {STREAM} · {time.perf_counter() - t0:.1f} s")

v10 · providers ['CPUExecutionProvider'] · face model on · streaming {'step_s': 0.5, 'lookahead_s': 0.45, 'stable_n': 1} · 2.1 s


In [3]:
import cv2


class Extractor:
    """Tier B: MediaPipe Holistic (same .task as the browser) · tier A: RTMW-X whole-body + signer tracking (modules/wholebody.py).
    frame(bgr, t_ms) → one raw frame (pose / lh / rh / face / bs [+ kp133 / sc133]); ms = extractor time per frame."""

    def __init__(self, tier=TIER):
        self.tier, self.ms = tier, []
        if tier == "A":
            from modules.wholebody import WholeBody
            self.m = WholeBody("x", DEVICE, fp16=True, redetect=10)
            print("tier A: RTMW-X on", self.m.prov[0], "(FP16)" if self.m.fp16 else "(FP32)")
        else:
            from modules.mediapipe_pose import Holistic
            self.m = Holistic(MODEL_DIR / "mediapipe" / "holistic_landmarker.task")

    def frame(self, bgr, t_ms):
        t = time.perf_counter()
        if self.tier == "A":
            f = self.m.frame(None, t_ms, bgr=bgr)
        else:
            pose, lh, rh, face, bs = self.m.frame(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), t_ms)
            f = dict(pose=pose, lh=lh, rh=rh, face=face, bs=bs)
        self.ms.append((time.perf_counter() - t) * 1000)
        return f

    def close(self):
        if self.tier != "A":
            self.m.close()


class Camera:
    """Webcam (camera clock) or a video file played at real time (so latency numbers mean the same)."""

    def __init__(self, source=SOURCE):
        self.cap = cv2.VideoCapture(source)
        self.file = not isinstance(source, int)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 25.0
        self.t0, self.n = time.perf_counter(), 0

    def read(self):
        ok, bgr = self.cap.read()
        if not ok:
            return None, None
        if self.file:                                  # pace the file like a camera
            t = self.n / self.fps; self.n += 1
            wait = self.t0 + t - time.perf_counter()
            if wait > 0:
                time.sleep(wait)
            return bgr, t
        return bgr, time.perf_counter() - self.t0

    def close(self):
        self.cap.release()


print("extractor + camera helpers ready")

extractor + camera helpers ready


In [4]:
import base64, io, urllib.request
import soundfile as sf
from IPython.display import Audio, display
from modules.speech import load_voice, plan_speech


class ServerVoice:
    """F5-TTS-THAI in the speech env (webapp/tts/app.py, started in cell 6.0): the forward result goes over, the plan is made there."""

    def __init__(self, url):
        self.url = url
        json.load(urllib.request.urlopen(url + "/v1/health", timeout=2))

    def speak_result(self, result, affect):
        body = dict(segments=[{k: v for k, v in s.items() if k in ("nearest", "status", "p_correct", "t0", "t1", "f0", "f1")} for s in result["segments"]],
                    affect={k: affect.get(k) for k in ("emotion", "intensity", "arousal")} if affect else {}, sentence=result.get("sentence", ""),
                    provider=(result.get("fusion") or {}).get("provider") or "rules", timings_ms=result.get("timings_ms", {}),
                    voice=VOICE, speak_uncertain=SPEAK_UNCERTAIN, disfluency=DISFLUENCY)
        req = urllib.request.Request(self.url + "/v1/speak_plan", data=json.dumps(body, ensure_ascii=False).encode(), headers={"Content-Type": "application/json"})
        r = json.load(urllib.request.urlopen(req, timeout=120))
        if not r["wav_b64"]:
            return np.zeros(0, np.float32), 24000, r["plan"]
        y, sr = sf.read(io.BytesIO(base64.b64decode(r["wav_b64"])), dtype="float32")
        return y, sr, r["plan"]


class LocalVoice:
    """The voice in this kernel: load_voice → CPU voice (MMS-TTS + the same fillers) when F5 is not installed / no GPU."""

    def __init__(self):
        self.v = load_voice(SPEECH_DIR, device="cpu" if SPEECH_ENGINE == "cpu" else None)

    def speak_result(self, result, affect):
        plan = plan_speech(result, affect, voice=VOICE, speak_uncertain=SPEAK_UNCERTAIN, disfluency=DISFLUENCY)
        y, sr = self.v.speak(plan)
        return y, sr, dict(text=plan.text, units=plan.units, emotion=plan.emotion, intensity=plan.intensity, reason=plan.reason)


def make_voice():
    if SPEECH_ENGINE in ("auto", "server"):
        try:
            v = ServerVoice(SPEECH_SERVER); print("เสียง: F5-TTS-THAI ผ่าน speech server", SPEECH_SERVER); return v
        except Exception as e:                        # noqa: BLE001
            if SPEECH_ENGINE == "server":
                raise
    v = LocalVoice(); print("เสียง:", type(v.v).__name__, "(ใน kernel นี้)"); return v


def say(result, affect=None, show=True):
    """Forward result → speech plan → audio. Returns (wav, sr, plan, ms) and shows a play button."""
    t = time.perf_counter()
    y, sr, plan = VOICE_ENGINE.speak_result(result, affect or {})
    ms = (time.perf_counter() - t) * 1000
    if show:
        print("   พูด:", " ".join(("[" + (u.get("filler") or "เว้น") + f" {u['pause_s']:.1f}s]") if u["kind"] == "hesitate" else u["text"]
                                for u in plan["units"]) if plan.get("units") else plan.get("text"), "|", " · ".join(plan.get("reason", [])))
        print(f"   เสียง {len(y) / max(sr, 1):.1f} s · สังเคราะห์ {ms:.0f} ms")
        if len(y):
            display(Audio(y, rate=sr))
    return y, sr, plan, ms


VOICE_ENGINE = make_voice()

c:\Users\BM MONEY\miniconda3\envs\490-sign\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


เสียง: CPUVoice (ใน kernel นี้)


## 3. ฟังก์ชันช่วย

* `hands_up` — กฎเดียวกับเว็บ: มือ (21 จุด) สูงกว่าไหล่ + 1.5×ความกว้างไหล่ หรือข้อมือของ pose สูงกว่า + 1.3×
* `draw` — วาด landmark + คำยืนยันแล้ว (ขาว) / partial (เทา) + fps / ms ต่อเฟรม (ภาพกระจกแบบเว็บ)
* `to_raw` — รวมเฟรมเป็น dict แบบเดียวกับ `extract()` / `from_browser()` / tier A

In [5]:
HAND_LINKS = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12),
              (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]
MARK = {"accepted": "✓", "uncertain": "[?]", "unknown": "✗", "transition": "~"}


def hands_up(pose, lh, rh):
    if pose[:, 3].max() <= 0:
        return False
    ls, rs = pose[11], pose[12]
    sw = float(np.hypot(ls[0] - rs[0], ls[1] - rs[1])) or 0.2
    mid_y = (ls[1] + rs[1]) / 2
    up = lambda h: np.abs(h).sum() > 0 and h[:, 1].mean() < mid_y + 1.5 * sw
    wrist = lambda i: pose[i, 3] > 0.5 and pose[i, 1] < mid_y + 1.3 * sw
    return bool(up(lh) or up(rh) or wrist(15) or wrist(16))


def to_raw(frames, hw):
    t = np.array([f["t"] for f in frames], np.float64); t = t - t[0]
    keys = [k for k in frames[0] if k != "t"]
    out = dict(t=t, hw=np.array(hw, np.int32), **{k: np.stack([f[k] for f in frames]).astype(np.float32) for k in keys},
               src_fps=np.float32(len(t) / max(t[-1], 1e-3)))
    if "kp133" in out:
        out["extractor"] = "rtmw"
    return out


def put_th(img, text, org, col, scale=0.8):
    """Thai text on an OpenCV image (cv2.putText cannot draw Thai): PIL with Tahoma / Leelawadee UI."""
    from PIL import Image, ImageDraw, ImageFont
    font = None
    for f in ("C:/Windows/Fonts/LeelawUI.ttf", "C:/Windows/Fonts/tahoma.ttf", "/System/Library/Fonts/Thonburi.ttc"):
        if Path(f).exists():
            font = ImageFont.truetype(f, int(32 * scale)); break
    im = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); d = ImageDraw.Draw(im)
    d.text(org, text, fill=col[::-1], font=font)
    return cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2BGR)


def draw(img, f, header, confirmed=(), partial=(), note=""):
    h, w = img.shape[:2]
    for hand, col in ((f["lh"], (80, 200, 120)), (f["rh"], (60, 140, 255))):
        if np.abs(hand).sum() > 0:
            for a, b in HAND_LINKS:
                cv2.line(img, (int(hand[a, 0] * w), int(hand[a, 1] * h)), (int(hand[b, 0] * w), int(hand[b, 1] * h)), col, 2)
    for i in (11, 12, 13, 14, 15, 16):
        if f["pose"][i, 3] > 0.5:
            cv2.circle(img, (int(f["pose"][i, 0] * w), int(f["pose"][i, 1] * h)), 4, (255, 255, 255), -1)
    img = cv2.flip(img, 1)
    cv2.rectangle(img, (0, h - 70), (w, h), (0, 0, 0), -1)
    img = put_th(img, " ".join(confirmed[-8:]), (12, h - 64), (255, 255, 255))
    if partial:
        img = put_th(img, " ".join(partial), (12, h - 30), (150, 150, 150), 0.7)
    cv2.putText(img, f"{header}  {note}", (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return img


def stats(ms):
    ms = np.asarray(ms, float)
    return f"p50 {np.percentile(ms, 50):.0f} ms · p95 {np.percentile(ms, 95):.0f} ms" if len(ms) else "-"


print("helpers ready")

helpers ready


## 4. Sign — real-time streaming (คำขึ้นระหว่างทำท่า)

ทำท่าต่อเนื่องได้เลย (ไม่ต้องวางมือระหว่างคำ): ทุก ~0.5 วิ โมเดลอ่านหน้าต่างล่าสุด (≤ 8 วิ) **ใน thread แยก** — กล้องไม่ค้าง
* คำ **เทา** = partial (ยังอาจเปลี่ยน) · คำ **ขาว** = ยืนยันแล้ว (ไม่แก้ย้อนหลัง) · วางมือ ≥ 0.8 วิ = ยืนยันที่เหลือทั้งหมด
* แสดงเป็น **รอบ**: ยกมือ = เริ่มรอบใหม่บนบรรทัดว่าง · วางมือ ≥ `END_MS` = จบรอบ — คำของรอบนั้นค้างบนจอ `ROUND_HOLD_S` วิแล้วล้าง (รอบไม่ปนกัน)
* หลังกด **q**: latency ต่อคำ = เวลาที่คำยืนยันขึ้นจอ − เวลาที่ท่านั้นจบ (นับเวลาคำนวณจริง), fps, ms/เฟรมของ extractor, flicker

In [12]:
from modules.stream import LiveStream

ROUND_HOLD_S = 3.0     # จบรอบแล้ว (วางมือ ≥ END_MS) คำของรอบนั้นค้างบนจออีกกี่วินาทีก่อนล้าง — รอบใหม่เริ่มบรรทัดว่างเสมอ

cam, ext = Camera(SOURCE), Extractor(TIER)
live, shown, n, t_last, fps = None, 0, 0, None, []
rounds, r_start, r_last_up, held = [], None, 0.0, None      # held = (start, end, show until) ของรอบที่เพิ่งจบ
try:
    while True:
        bgr, t = cam.read()
        if bgr is None:
            break
        f = ext.frame(bgr, t * 1000.0)
        if live is None:
            live = LiveStream(pipe, bgr.shape[:2], alphabet=ALPHABET, **STREAM)
        live.add(f, t)
        if t_last is not None:
            fps.append(1.0 / max(t - t_last, 1e-3))
        t_last = t
        if hands_up(f["pose"], f["lh"], f["rh"]):                  # ยกมือ: รอบใหม่ (ถ้ายังไม่ได้อยู่ในรอบ)
            if r_start is None:
                r_start, held = t - LIVE_PREROLL_MS / 1000, None
                print(f"── รอบ {len(rounds) + 1} ──")
            r_last_up = t
        elif r_start is not None and t - r_last_up >= END_MS / 1000:   # วางมือนานพอ: จบรอบ
            rounds.append((r_start, r_last_up)); held = (r_start, r_last_up, t + ROUND_HOLD_S); r_start = None
        while shown < len(live.confirmed):
            c = live.confirmed[shown]; shown += 1
            print(f"✓ {c['word']:<12} {c['t0']:6.2f}–{c['t1']:5.2f} s  latency {c['latency']:.2f} s  p={c['p']:.2f}")
        if SHOW_WINDOW:
            if r_start is not None:                                   # กำลังทำท่า: เฉพาะคำของรอบนี้
                conf = [c["word"] for c in live.confirmed if c["t1"] > r_start]
                part = [s["word"] for s in live.partial_segments if (s["t0"] + s["t1"]) / 2 > r_start]
            elif held and t < held[2]:                                # เพิ่งจบรอบ: คำของรอบนั้นอีกครู่หนึ่ง
                conf, part = [c["word"] for c in live.confirmed if held[0] < c["t1"] <= held[1] + 0.5], []
            else:                                                     # ไม่ได้ทำท่า: ล้างจอ
                conf, part = [], []
            cv2.imshow("real-time-lab · sign", draw(bgr.copy(), f, f"SIGN {np.mean(fps[-30:]) if fps else 0:.0f} fps", conf, part,
                                                   f"round {len(rounds) + (r_start is not None)} · extractor {ext.ms[-1]:.0f} ms"))
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
except KeyboardInterrupt:
    pass
finally:
    words = live.close() if live else []
    cam.close(); ext.close()
    if SHOW_WINDOW:
        cv2.destroyAllWindows(); cv2.waitKey(1)
if r_start is not None:
    rounds.append((r_start, t_last))
lat = [c["latency"] for c in words]
for i, (a, b) in enumerate(rounds, 1):
    print(f"รอบ {i} ({a:.1f}–{b:.1f} s):", " ".join(c["word"] for c in words if a < c["t1"] <= b + 0.5) or "-")
print("\nคำที่ยืนยันทั้งหมด:", " ".join(c["word"] for c in words) or "-")
if live:
    print(f"latency ต่อคำ p50 {np.percentile(lat, 50) if lat else float('nan'):.2f} s · p95 {np.percentile(lat, 95) if lat else float('nan'):.2f} s"
          f" | compute ต่อรอบ {stats(live.st.proc_ms)} | flicker {live.st.flicker} | fps {np.median(fps) if fps else 0:.1f}"
          f" | extractor ({'MediaPipe' if TIER != 'A' else 'RTMW-X'}) {stats(ext.ms)}")
SIGN_RESULT = dict(words=[c["word"] for c in words], rounds=[[c["word"] for c in words if a < c["t1"] <= b + 0.5] for a, b in rounds],
                   latency=lat, proc_ms=live.st.proc_ms if live else [], extractor_ms=ext.ms, fps=fps)

Exception in thread Thread-10 (_run):
Traceback (most recent call last):
  File "c:\Users\BM MONEY\miniconda3\envs\490-sign\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\BM MONEY\miniconda3\envs\490-sign\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "d:\SLM-labs\SLM-Labs-SignDINO\modules\stream.py", line 156, in _run
    self.st._step(now)
  File "d:\SLM-labs\SLM-Labs-SignDINO\modules\stream.py", line 73, in _step
    r = self.pipe.run_landmarks(raw, alphabet=self.alphabet)
  File "d:\SLM-labs\SLM-Labs-SignDINO\modules\pipeline.py", line 553, in run_landmarks
    face = analyse(raw, [(s["f0"], s["f1"]) for s in segs if s["status"] != "transition"], baseline_bs=face_baseline)
  File "d:\SLM-labs\SLM-Labs-SignDINO\modules\face.py", line 125, in analyse
    head_shake_cycles=_cycles(yaw, 0.06), head_nod_cycles=_cycles(pitch, 0.05),
  File "d:\SLM-labs\SLM-Labs-SignDINO\modules\face.py", line 53, in _cycles
    

── รอบ 1 ──
── รอบ 2 ──
── รอบ 3 ──
── รอบ 4 ──
── รอบ 5 ──
รอบ 1 (5.3–6.9 s): -
รอบ 2 (12.1–14.6 s): -
รอบ 3 (17.6–20.8 s): -
รอบ 4 (25.0–26.8 s): -
รอบ 5 (29.2–31.1 s): ฉัน

คำที่ยืนยันทั้งหมด: ฉัน
latency ต่อคำ p50 5.79 s · p95 5.79 s | compute ต่อรอบ p50 842 ms · p95 842 ms | flicker 0 | fps 21.4 | extractor (MediaPipe) p50 30 ms · p95 37 ms


## 5. Face — อารมณ์จากสีหน้าแบบสด (ไม่ต้องทำท่า)

ทุก ~1 วิ อ่านสีหน้าจาก 2 วิล่าสุด ด้วย `models/face-model` (MediaPipe face blendshapes, อธิบายได้):
อารมณ์ 7 แบบ + ความมั่นใจ + เหตุผล (เช่น "มุมปากยกขึ้น (ยิ้ม)") · ใช้ MediaPipe เสมอ (RTMW ไม่มี blendshape)

In [7]:
from modules.face_model import EMO_TH, describe

assert pipe.face_model is not None, "ไม่พบ models/face-model"
cam, ext = Camera(SOURCE), Extractor("B")
buf, last, rows, label = [], -1.0, [], "-"
try:
    while True:
        bgr, t = cam.read()
        if bgr is None:
            break
        f = ext.frame(bgr, t * 1000.0)
        H, W = bgr.shape[:2]
        face = f["face"].copy(); face[:, 0] *= W / H
        buf.append((t, f["bs"], face)); buf = [b for b in buf if t - b[0] <= 2.0]
        if t - last >= 1.0 and len(buf) >= 10:
            last = t
            bs = np.stack([b[1] for b in buf]); fc = np.stack([b[2] for b in buf])
            r = pipe.face_model.predict_stretch(bs, fc)
            r.update(emotion_th=EMO_TH[r["emotion"]], confidence=r["p"])
            label = describe(r)
            rows.append(dict(t=round(t, 1), emotion=r["emotion"], p=round(r["p"], 2), valence=round(r["valence"], 2), why=label))
            print(f"{t:6.1f}s  {label}")
        if SHOW_WINDOW:
            img = cv2.flip(bgr.copy(), 1)
            img = put_th(img, label, (12, 12), (0, 255, 255), 0.7)
            cv2.imshow("real-time-lab · face", img)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
except KeyboardInterrupt:
    pass
finally:
    cam.close(); ext.close()
    if SHOW_WINDOW:
        cv2.destroyAllWindows(); cv2.waitKey(1)
import pandas as pd
FACE_ROWS = pd.DataFrame(rows); FACE_ROWS

   1.4s  สีหน้าเป็นกลาง (ความมั่นใจ 87%)
   2.4s  สีหน้าเป็นกลาง (ความมั่นใจ 87%)
   3.5s  สีหน้าเป็นกลาง (ความมั่นใจ 35%)
   4.5s  สีหน้าประหลาดใจ / ตกใจ (ความมั่นใจ 47%) เพราะหลับตาซ้าย · หางคิ้วขวายก
   5.5s  สีหน้าดีใจ / มีความสุข (ความมั่นใจ 56%) เพราะมุมปากขวายก (ยิ้ม) · มุมปากซ้ายยก (ยิ้ม) · ยื่นคาง
   6.5s  สีหน้าดีใจ / มีความสุข (ความมั่นใจ 84%) เพราะมุมปากขวายก (ยิ้ม) · มุมปากซ้ายยก (ยิ้ม) · ยกริมฝีปากบน
   7.6s  สีหน้าดีใจ / มีความสุข (ความมั่นใจ 81%) เพราะมุมปากขวายก (ยิ้ม) · มุมปากซ้ายยก (ยิ้ม) · ยกริมฝีปากบน
   8.6s  สีหน้าดีใจ / มีความสุข (ความมั่นใจ 76%) เพราะมุมปากขวายก (ยิ้ม) · มุมปากซ้ายยก (ยิ้ม) · ยกริมฝีปากบน
   9.6s  สีหน้าดีใจ / มีความสุข (ความมั่นใจ 62%) เพราะมุมปากขวายก (ยิ้ม) · มุมปากซ้ายยก (ยิ้ม) · ยกริมฝีปากบน
  10.6s  สีหน้าดีใจ / มีความสุข (ความมั่นใจ 26%) เพราะมุมปากขวายก (ยิ้ม) · ยื่นคาง · ยกริมฝีปากบน
  11.6s  สีหน้าเป็นกลาง (ความมั่นใจ 28%)
  12.7s  สีหน้าเป็นกลาง (ความมั่นใจ 34%)
  13.7s  สีหน้าเป็นกลาง (ความมั่นใจ 43%)
  14.7s  สีหน้าเป็นกลาง (ความมั

,t,emotion,p,valence,why
0,1.4,neutral,0.87,-0.03,สีหน้าเป็นกลาง (ความมั่นใจ 87%)
1,2.4,neutral,0.87,-0.03,สีหน้าเป็นกลาง (ความมั่นใจ 87%)
2,3.5,neutral,0.35,-0.15,สีหน้าเป็นกลาง (ความมั่นใจ 35%)
3,4.5,surprise,0.47,0.07,สีหน้าประหลาดใจ / ตกใจ (ความมั่นใจ 47%) เพราะห...
4,5.5,happy,0.56,0.33,สีหน้าดีใจ / มีความสุข (ความมั่นใจ 56%) เพราะม...
5,6.5,happy,0.84,0.60,สีหน้าดีใจ / มีความสุข (ความมั่นใจ 84%) เพราะม...
6,7.6,happy,0.81,0.56,สีหน้าดีใจ / มีความสุข (ความมั่นใจ 81%) เพราะม...
7,8.6,happy,0.76,0.49,สีหน้าดีใจ / มีความสุข (ความมั่นใจ 76%) เพราะม...
8,9.6,happy,0.62,0.33,สีหน้าดีใจ / มีความสุข (ความมั่นใจ 62%) เพราะม...
9,10.6,happy,0.26,-0.11,สีหน้าดีใจ / มีความสุข (ความมั่นใจ 26%) เพราะม...


## 6. Speech — เสียงจากโมเดล speech v2 (รวมการลังเลแบบมนุษย์)

**6.0 (ไม่บังคับ) เปิด speech server สำหรับเสียง F5-TTS-THAI** — kernel นี้ไม่มี F5 จึงรันใน speech env (`D:/SLM-labs/envs/speech`, ใช้ GPU)
ถ้าไม่เปิด จะใช้เสียง CPU (MMS-TTS) ในเครื่องนี้แทน — แผนการพูดและคำลังเลเหมือนกัน · ปิด server ที่ cell สุดท้าย

In [8]:
import subprocess
SPEECH_PY = Path("D:/SLM-labs/envs/speech/Scripts/python.exe")
if SPEECH_PY.exists():
    speech_srv = subprocess.Popen([str(SPEECH_PY), "-m", "uvicorn", "webapp.tts.app:app", "--host", "127.0.0.1", "--port", SPEECH_SERVER.rsplit(":", 1)[1]],
                                  cwd=str(ROOT), env={**os.environ, "SPEECH_MODEL": str(SPEECH_DIR), "SPEECH_DEVICE": "auto", "HF_HUB_CACHE": "D:/SLM-labs/hf_cache"},
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(120):                                  # F5 + vocoder load ≈ 20–40 s
        try:
            print(json.load(urllib.request.urlopen(SPEECH_SERVER + "/v1/health", timeout=1))); break
        except Exception:                                 # noqa: BLE001
            time.sleep(1)
    VOICE_ENGINE = make_voice()
else:
    print("ไม่พบ speech env — ใช้เสียง CPU ต่อไป")

{'ok': True, 'voice': 'facebook/mms-tts-tha', 'plan_voice': 'ExpressiveTTS', 'speech_model': 'speech-model-v2'}
เสียง: F5-TTS-THAI ผ่าน speech server http://127.0.0.1:8011


**6.1 ประโยคทดสอบ** — `?` = ท่าที่โมเดลอ่านไม่ได้ (`[?]`). v9 ข้ามไปเลย ("ฉัน พ่อ"), v10 เว้นจังหวะ/ลังเลแบบคน ("ฉัน… อืม… พ่อ")
โดย **ไม่พูดคำเดา** · `?` ท้ายประโยค = เงียบ · หลาย `?` ติดกัน = ลังเลครั้งเดียว · เปลี่ยน `DISFLUENCY` ใน cell 1 เพื่อเทียบ

In [9]:
def fake_result(words, sign_s=0.7):
    """'ฉัน ? พ่อ' → a forward result: accepted words, '?' = an unread sign (its nearest word is never spoken)."""
    segs, t = [], 0.0
    for w in words.split():
        segs.append(dict(nearest="รถไฟ" if w == "?" else w, status="unknown" if w == "?" else "accepted", p_correct=0.1 if w == "?" else 0.95,
                         t0=t, t1=t + sign_s, f0=int(t * 25), f1=int((t + sign_s) * 25)))
        t += sign_s + 0.1
    return dict(segments=segs, fusion=dict(provider="rules"), sentence="")


for text in ["ฉัน ? พ่อ", "? ฉัน", "ฉัน ? ? กิน", "พรุ่งนี้ ฉัน ? ไป โรงเรียน", "แม่ ? ชอบ ? ขนมปัง", "สวัสดี ฉัน ชื่อ ?"]:
    print(f"\n{text}")
    say(fake_result(text))


ฉัน ? พ่อ
   พูด: ฉัน [um 0.3s] พ่อ | คำตามลำดับที่ทำท่า (ยืนยันแล้ว) · ท่าที่อ่านไม่ได้ 1 ท่า → เว้นจังหวะ/คำลังเล (ไม่เดาคำ) · เสียงต้นแบบ: female/neutral
   เสียง 2.4 s · สังเคราะห์ 9223 ms



? ฉัน
   พูด: [uh 0.3s] ฉัน | คำตามลำดับที่ทำท่า (ยืนยันแล้ว) · ท่าที่อ่านไม่ได้ 1 ท่า → เว้นจังหวะ/คำลังเล (ไม่เดาคำ) · เสียงต้นแบบ: female/neutral
   เสียง 1.6 s · สังเคราะห์ 3062 ms



ฉัน ? ? กิน
   พูด: ฉัน [uh_long 0.5s] กิน | คำตามลำดับที่ทำท่า (ยืนยันแล้ว) · ท่าที่อ่านไม่ได้ 2 ท่า → เว้นจังหวะ/คำลังเล (ไม่เดาคำ) · เสียงต้นแบบ: female/neutral
   เสียง 2.5 s · สังเคราะห์ 6159 ms



พรุ่งนี้ ฉัน ? ไป โรงเรียน
   พูด: พรุ่งนี้ ฉัน [uh 0.3s] ไป โรงเรียน | คำตามลำดับที่ทำท่า (ยืนยันแล้ว) · ท่าที่อ่านไม่ได้ 1 ท่า → เว้นจังหวะ/คำลังเล (ไม่เดาคำ) · เสียงต้นแบบ: female/neutral
   เสียง 2.5 s · สังเคราะห์ 6221 ms



แม่ ? ชอบ ? ขนมปัง
   พูด: แม่ [uh 0.3s] ชอบ [um 0.3s] ขนมปัง | คำตามลำดับที่ทำท่า (ยืนยันแล้ว) · ท่าที่อ่านไม่ได้ 2 ท่า → เว้นจังหวะ/คำลังเล (ไม่เดาคำ) · เสียงต้นแบบ: female/neutral
   เสียง 4.0 s · สังเคราะห์ 9043 ms



สวัสดี ฉัน ชื่อ ?
   พูด: สวัสดี ฉัน ชื่อ | คำตามลำดับที่ทำท่า (ยืนยันแล้ว) · เสียงต้นแบบ: female/neutral
   เสียง 1.3 s · สังเคราะห์ 2985 ms


In [10]:
SAY = "วันนี้ ? เพื่อน ? ? ร้องไห้"     # พิมพ์ประโยคเอง (? = ท่าที่อ่านไม่ได้)
for mode in ("skip", "pause", "filler"):
    DISFLUENCY, keep = mode, DISFLUENCY
    print(f"\n[{mode}]"); say(fake_result(SAY))
    DISFLUENCY = keep


[skip]
   พูด: วันนี้ เพื่อน ร้องไห้ | คำตามลำดับที่ทำท่า (ยืนยันแล้ว) · เสียงต้นแบบ: female/neutral
   เสียง 1.3 s · สังเคราะห์ 2799 ms



[pause]


KeyboardInterrupt: 

## 7. Overall — production test (Sign + Face + Speech)

ยกมือ ≥ 0.25 วิ = เริ่มประโยค (เก็บ 0.5 วิก่อนหน้า) · ระหว่างทำท่าคำขึ้นแบบ streaming · **วางมือ ≥ 1.5 วิ = จบประโยค** →
โมเดลอ่านทั้งประโยคอีกครั้ง (แม่นสุด) → คำ + สถานะ + ประโยค + อารมณ์ (หน้า + คำ + พลังท่ามือ) → แผนการพูด → **▶ ปุ่มเล่นเสียง**

In [11]:
from modules.face_model import fuse_affect, signing_energy

OVERALL = []


def finish_utterance(frames, hw, k):
    t = time.perf_counter()
    r = pipe.run_landmarks(to_raw(frames, hw), title=f"utt{k}", alphabet=ALPHABET)
    model_ms = (time.perf_counter() - t) * 1000
    aff = (r.get("face") or {}).get("affect") or {}
    print(f"\n#{k}  ท่า {r['duration_s']} s → โมเดล {model_ms:.0f} ms (" + ", ".join(f"{a} {b:.0f}" for a, b in r["timings_ms"].items() if a.endswith("_ms")) + ")")
    for s in r["segments"]:
        if s["status"] != "transition":
            alt = "" if s["status"] == "accepted" else "   ตัวเลือก: " + " · ".join(c["word"] for c in s["candidates"][:5])
            print(f"   {MARK.get(s['status'], '?'):>3} {s['nearest']:<14} p={s.get('p_correct', 0):.2f} ({s['status']}){alt}")
    print("   ประโยค:", r.get("sentence") or "(ยังไม่มั่นใจ)", "| อารมณ์:", aff.get("explanation_th") or (r.get("face") or {}).get("emotion_th"))
    y, sr, plan, speech_ms = say(r, aff)
    OVERALL.append(dict(utt=k, words=[(s["nearest"], s["status"]) for s in r["segments"] if s["status"] != "transition"], sentence=r.get("sentence"),
                        emotion=aff.get("emotion"), said=plan.get("text"), model_ms=round(model_ms), speech_ms=round(speech_ms), audio_s=round(len(y) / max(sr, 1), 2)))


cam, ext = Camera(SOURCE), Extractor(TIER)
state, pre, utt, up_since, start, last_up, k, live = "waiting", [], [], 0.0, 0.0, 0.0, 0, None
try:
    while True:
        bgr, t = cam.read()
        if bgr is None:
            break
        f = ext.frame(bgr, t * 1000.0); f["t"] = t
        up = hands_up(f["pose"], f["lh"], f["rh"])
        hw = bgr.shape[:2]
        if state == "waiting":
            pre.append(f); pre = [x for x in pre if t - x["t"] <= LIVE_PREROLL_MS / 1000]
            up_since = (up_since or t) if up else 0.0
            if up_since and t - up_since >= LIVE_UP_MS / 1000:
                state, utt, start, last_up = "signing", list(pre), t, t
                live = LiveStream(pipe, hw, alphabet=ALPHABET, **STREAM)
                for x in utt:
                    live.add({kk: v for kk, v in x.items() if kk != "t"}, x["t"])
        else:
            utt.append(f); live.add({kk: v for kk, v in f.items() if kk != "t"}, t)
            if up:
                last_up = t
            if t - last_up >= END_MS / 1000 or t - start >= LIVE_MAX_MS / 1000:
                frames = [x for x in utt if x["t"] <= last_up + 0.4]
                live.close(flush=False); live = None
                state, pre, up_since = "waiting", [], 0.0
                dur = frames[-1]["t"] - frames[0]["t"] if len(frames) > 1 else 0
                up_s = sum(b["t"] - a["t"] for a, b in zip(frames, frames[1:]) if hands_up(b["pose"], b["lh"], b["rh"]))
                if dur * 1000 < LIVE_MIN_MS or up_s * 1000 < LIVE_MIN_UP_MS:
                    print(f"ข้ามการขยับสั้น ๆ ({dur:.1f} วิ) — ไม่ใช่ท่าภาษามือ")
                else:
                    if SHOW_WINDOW:
                        cv2.imshow("real-time-lab · overall", draw(bgr.copy(), f, "PROCESSING")); cv2.waitKey(1)
                    k += 1
                    finish_utterance(frames, hw, k)
        if SHOW_WINDOW:
            conf = [c["word"] for c in live.confirmed] if live else []
            cv2.imshow("real-time-lab · overall", draw(bgr.copy(), f, state.upper(), conf, live.partial if live else [],
                                                       f"{t - start:.1f}s" if state == "signing" else ""))
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
except KeyboardInterrupt:
    pass
finally:
    if state == "signing" and len(utt) > 10:
        k += 1; finish_utterance(utt, hw, k)
    if live:
        live.close(flush=False)
    cam.close(); ext.close()
    if SHOW_WINDOW:
        cv2.destroyAllWindows(); cv2.waitKey(1)
print(f"\nจบ: {k} ประโยค · extractor {stats(ext.ms)}")


#1  ท่า 3.68 s → โมเดล 345 ms (schema_ms 8, tagger_ms 201, spotting_ms 127, face_ms 8, language_ms 0)
     ✗ ฉัน            p=0.17 (unknown)   ตัวเลือก: ฉัน · ลูก · ข้าว · กิน · ยาย
     ✗ กิน            p=0.07 (unknown)   ตัวเลือก: กิน · ยาย · สิบ · ไป · ย่า
   ประโยค: (ยังไม่มั่นใจ) | อารมณ์: สีหน้าเป็นกลาง (ความมั่นใจ 52%)
   พูด:  | คำตามลำดับที่ทำท่า (ยืนยันแล้ว)
   เสียง 0.0 s · สังเคราะห์ 3 ms

#2  ท่า 20.52 s → โมเดล 2120 ms (schema_ms 48, tagger_ms 1121, spotting_ms 934, face_ms 13, language_ms 0)
   [?] กิน            p=0.34 (uncertain)   ตัวเลือก: กิน · ฉัน · ข้าว · เกิด · ขนมปัง
   [?] ลูก            p=0.21 (uncertain)   ตัวเลือก: ลูก · ฉัน · กิน · วันนี้ · กรุงเทพ
   [?] คุณ            p=0.35 (uncertain)   ตัวเลือก: คุณ · เที่ยว · แฟน · กิน · ข้าว
   [?] ชื่อ           p=0.81 (uncertain)   ตัวเลือก: ชื่อ · วิ่ง · ไข่ · หลาน · นั่ง
   [?] กิน            p=0.24 (uncertain)   ตัวเลือก: กิน · ฉัน · ข้าว · เกิด · ขนมปัง
   [?] ฉัน            p=0.34 (uncertain)   ตัวเลือก: ฉัน 

## 8. สรุป + บันทึก (ไม่มีภาพ เก็บแค่ผล)

In [ ]:
import pandas as pd
display(pd.DataFrame(OVERALL))
out = ROOT / "cache" / "real_time_lab"; out.mkdir(parents=True, exist_ok=True)
p = out / time.strftime("%Y%m%d_%H%M%S.json")
p.write_text(json.dumps(dict(model=MODEL_DIR.name, tier=TIER, stream=STREAM, overall=OVERALL,
                             sign={k: v for k, v in (SIGN_RESULT if "SIGN_RESULT" in globals() else {}).items()}), ensure_ascii=False, indent=1, default=float),
             encoding="utf-8")
print(p)

In [ ]:
# ปิด speech server (ถ้าเปิดใน 6.0) — ไม่ให้ค้างกิน GPU / หน่วยความจำ
if "speech_srv" in globals() and speech_srv.poll() is None:
    speech_srv.terminate(); speech_srv.wait(10); print("speech server stopped")